# Notebook Purpose

The purpose of this notebook is to replicate the GCN Benchmark results.

## Imports

In [ ]:
import subprocess
import psutil
import random
import time
import statistics
from pathlib import Path
from datetime import datetime, timezone, timedelta

import humanize
import GPUtil as GPU
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import trange
from dgl.dataloading import GraphDataLoader
from dgl.nn import GraphConv, SumPooling
import dgl.function as fn
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

from datasets.dataset_handler import DatasetHandler
from datasets.enums import FeatureType, DatasetSource

## Attributes

In [ ]:
handler = DatasetHandler()
feature_types = [FeatureType.DGL, FeatureType.DGL_WITH_BONDS]
dataset_sources = list(DatasetSource)

## Methods

### Retrieve CMD Output

Method to run and retrieve results from the CMD process.

In [ ]:
def get_cmd_output(command):
    return subprocess.check_output(
        command,
        stderr=subprocess.STDOUT,
        shell=True
    ).decode('UTF-8')

## Main Logic

In [ ]:
cpu = get_cmd_output('cat /proc/cpuinfo | grep -E "model name"')
cpu = cpu.split('\n')[0].split('\t: ')[-1]
physical_cpu_count = psutil.cpu_count(logical=False)
logical_cpu_count = psutil.cpu_count() # physical count X no. of threads per physical core
try:
    cuda_version = get_cmd_output('nvcc --version | grep -E "Build"')
except subprocess.CalledProcessError:
    cuda_version = "Not Available"
try:
    gpu = get_cmd_output("nvidia-smi -L")
except subprocess.CalledProcessError:
    gpu = "Not Available"
general_ram_gb = humanize.naturalsize(psutil.virtual_memory().available)
try:
    gpu_ram_total_mb = GPU.getGPUs()[0].memoryTotal
except IndexError:
    gpu_ram_total_mb = "Not Available"

print(f"CPU: '{cpu}'")
print(f"Physical CPU Count: '{physical_cpu_count}'")
print(f"Logical CPU Count: '{logical_cpu_count}'")
print(f"CUDA Version: '{cuda_version}'")
print(f"GPU: '{gpu}'")
print(f"Available RAM: '{general_ram_gb}'")
print(f"GPU RAM: '{gpu_ram_total_mb}'")

### Parameters

In [ ]:
rounds = 20
episodes = 2000
learning_rate = 0.001
randomseed = 12
torch.manual_seed(randomseed) 
np.random.seed(randomseed)
random.seed(randomseed)
torch.cuda.manual_seed(randomseed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.is_available()
torch.backends.cudnn.benchmark = False # selects fastest conv algo
torch.backends.cudnn.deterministic = True
ml_model = "gcn_benchmark"

combinations = [
    [10, 10],
    [5, 10], 
    [1, 10], 
    [1, 5], 
    [1, 1]
]

save_columns = [
    'EXPERIMENT_DATE',
    'CPU',
    'CPU COUNT',
    'GPU',
    'GPU RAM',
    'RAM',
    'CUDA',
    'DATASET SOURCE',
    'FEATURE TYPE',
    'MODEL ARCHITECTURE',
    'NUMBER POSITIVE',
    'NUMBER NEGATIVE',
    'TARGET',
    'ACCURACY',
    'ROC',
    'PRC',
    'TRAIN ROC',
    'TRAIN PRC',
    'EPISODES',
    'TRAINING TIME',
    'ROC VALUES',
    'PRC VALUES'
]

### Graph Embedding

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_channels, out_channels=128):
      super(GCN, self).__init__()
      self.conv1 = GraphConv(in_channels, 64)
      self.conv2 = GraphConv(64, 128)
      self.conv3 = GraphConv(128, 64)
      self.sum_pool = SumPooling()
      self.dense = nn.Linear(64, out_channels)

      self.dense2 = nn.Linear(128, 1)

    def forward(self, graph, in_feat):
        h = self.conv1(graph, in_feat)
        h = F.relu(h)
        graph.ndata['h'] = h       
        graph.update_all(fn.copy_u('h', 'm'), fn.max('m', 'h'))
      
        h = self.conv2(graph, graph.ndata['h'])
        h = F.relu(h)
        graph.ndata['h'] = h
        graph.update_all(fn.copy_u('h', 'm'), fn.max('m', 'h'))

        h = self.conv3(graph, graph.ndata['h'])
        h = F.relu(h)
        graph.ndata['h'] = h
        graph.update_all(fn.copy_u('h', 'm'), fn.max('m', 'h'))

        output = self.sum_pool(graph, graph.ndata['h'])
        output = torch.tanh(output)
        output = self.dense(output)
        output = torch.tanh(output)

        output = self.dense2(output)

        return output

### Training Loop

In [ ]:
def run_training_loop(train_X, train_y, episodes, learning_rate, feature_type):
    start_time = time.time()

    node_feat_size = 177
    embedding_size = 128

    encoder = GCN(node_feat_size, embedding_size)
    loss_fn = nn.BCEWithLogitsLoss()
    sig = nn.Sigmoid()

    if torch.cuda.is_available(): 
        encoder = encoder.cuda()
        sig = sig.cuda()
        loss_fn = loss_fn.cuda()

    encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr = learning_rate)
    encoder_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(encoder_optimizer, patience=50)

    episode_num = 1
    losses = []

    running_loss = 0.0
    running_acc = 0.0
    running_roc = []
    running_prc = []

    encoder.train()

    # for task in shuffled_train_tasks:
    pbar = trange(episodes, desc=f"Training")
    # while episode_num < episodes and not early_stop:
    for episode in pbar:
        episode_loss = []
        episode_acc = []
        episode_roc = []
        episode_prc = []

        dataloader = GraphDataLoader(train_X, batch_size=len(train_X), shuffle=False, pin_memory=True)
        for batched_graph in dataloader:
            batched_graph = batched_graph.to(device)
    
            if feature_type == FeatureType.DGL:
                pred = encoder.forward(batched_graph, batched_graph.ndata['feats'].float())
            elif feature_type == FeatureType.DGL_WITH_BONDS:
                pred = encoder.forward(batched_graph, (batched_graph.ndata['feats'].float(), batched_graph.edata['edge_feats'].float()))
            else:
                raise Exception("Unsupported Feature Type")
      
        labels = torch.FloatTensor(train_y)
        labels = labels.to(device)
        loss = loss_fn(pred.squeeze(), labels)
          
        pred = sig(pred)
        _, y_hat = pred.max(1)
    
        episode_loss.append(loss.item())
        encoder.zero_grad()
        loss.backward()
        encoder_optimizer.step()
    
        labels = labels.detach().cpu()
        y_hat = y_hat.detach().cpu()
        
        roc = roc_auc_score(labels, y_hat)
        prc = average_precision_score(labels, y_hat)
        acc = accuracy_score(labels, y_hat)
    
        running_roc.append(roc)
        running_prc.append(prc)
    
        losses.append(statistics.mean(episode_loss))
        pbar.set_description(f"Episode {episode_num} - Loss {statistics.mean(episode_loss):.6f} - LR {encoder_optimizer.param_groups[0]['lr']}")
        pbar.refresh()
    
        if encoder_optimizer.param_groups[0]['lr'] < 0.000001:
            break # early stop
        elif episode_num < episodes:
            episode_num += 1

        encoder_scheduler.step(loss)

    end_time = time.time()
    train_info = {
        "losses": losses,
        "duration": str(timedelta(seconds=(end_time - start_time))),
        "episodes": episode_num,
        "train_roc": statistics.mean(running_roc),
        "train_prc": statistics.mean(running_prc),
    }

    return encoder, losses, train_info

### Testing Loop

In [ ]:
def evaluate_test(encoder, test_X, test_y):
    encoder.eval()
    with torch.no_grad():
        dataloader = GraphDataLoader(test_X, batch_size=len(test_X), shuffle=False, pin_memory=True)
        for batched_graph in dataloader:
            batched_graph = batched_graph.to(device)

            if feature_type == FeatureType.DGL:
                pred = encoder.forward(batched_graph, batched_graph.ndata['feats'].float())
            elif feature_type == FeatureType.DGL_WITH_BONDS:
                pred = encoder.forward(batched_graph, (batched_graph.ndata['feats'].float(), batched_graph.edata['edge_feats'].float()))
            else:
                raise Exception("Unsupported Feature Type")
        # PRED
        pred = nn.Sigmoid()(pred)

        # _, y_hat_actual = pred.max(1)
        y_hat_actual = np.round(pred.detach().cpu())
        y_hat = pred
        # _, y_hat = pred.max(1)

        y_hat = y_hat.detach().cpu()
        labels = test_y
        y_hat_actual = y_hat_actual.detach().cpu()
        
        roc = roc_auc_score(labels, y_hat)
        prc = average_precision_score(labels, y_hat)
        acc = accuracy_score(labels, y_hat_actual)
        
    return labels, y_hat, roc, prc

### Run

In [ ]:
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

combinations = [
    [10, 10],
    [5, 10], 
    [1, 10], 
    [1, 5], 
    [1, 1]
]

for dataset_source in dataset_sources:
    dataset_source_val = dataset_source.value

    result_df = pd.DataFrame(columns=save_columns) # initialise the results DataFrame
    results_csv = results_dir / f"{ml_model}/{dataset_source_val}.csv"
    results_csv.parent.mkdir(parents=True, exist_ok=True)

    if results_csv.exists():
        print(f"Skipping GCN Benchmark experiment for '{dataset_source_val}' dataset. Results already exist at '{results_csv}'")
        continue
    
    for feature_type in feature_types:
        feature_type_val = feature_type.value
        train_dfs, test_dfs = handler.load_train_test_set(dataset_source=dataset_source, feature_type=feature_type)
        for no_pos, no_neg in combinations:
            for target in test_dfs.keys():
                dt_run_start = datetime.now(timezone.utc).strftime("%d/%m/%Y %H:%M:%S")
                print(f"\n{dt_run_start} - Running GCN benchmark experiment for the '{dataset_source_val}' dataset, '{feature_type_val}' feature type with '{no_pos}' positives and '{no_neg}' negatives for '{target}' target")

                running_roc = []
                running_prc = []
                start_time = time.time()

                for r in trange(rounds):
                    test_df = test_dfs[target]
                    support_neg = test_df[test_df['y'] == 0].sample(no_neg)
                    support_pos = test_df[test_df['y'] == 1].sample(no_pos)

                    train_data = pd.concat([support_neg, support_pos])
                    test_data = test_df.drop(train_data.index)

                    train_data = train_data.sample(frac=1)
                    test_data = test_data.sample(frac=1)

                    train_X, train_y = list(train_data['mol'].to_numpy()), train_data['y'].to_numpy(dtype=np.int16)
                    test_X, test_y = list(test_data['mol'].to_numpy()), test_data['y'].to_numpy(dtype=np.int16)

                    encoder, losses, train_info = run_training_loop(train_X, train_y, episodes, learning_rate, feature_type)
                    targets, preds, roc, prc = evaluate_test(encoder, test_X, test_y)

                    running_roc.append(roc)
                    running_prc.append(prc)

                end_time = time.time()
                duration = str(timedelta(seconds=(end_time - start_time)))
                
                rounds_roc = f"{statistics.mean(running_roc):.3f} \u00B1 {statistics.stdev(running_roc):.3f}"
                rounds_prc = f"{statistics.mean(running_prc):.3f} \u00B1 {statistics.stdev(running_prc):.3f}"
                rounds_rec = pd.DataFrame([
                    [
                        dt_run_start,
                        cpu,
                        logical_cpu_count,
                        gpu,
                        gpu_ram_total_mb,
                        general_ram_gb,
                        cuda_version,
                        dataset_source_val,
                        feature_type_val,
                        ml_model,
                        no_pos,
                        no_neg,
                        target,
                        0,
                        rounds_roc,
                        rounds_prc,
                        train_info["train_roc"],
                        train_info["train_prc"],
                        train_info["episodes"],
                        train_info["duration"],
                        running_roc,
                        running_prc
                    ]],
                    columns=save_columns
                 )
                result_df = pd.concat([result_df, rounds_rec])
        
    print(f"Saving experiment results to CSV at '{results_csv}'")
    result_df.to_csv(results_csv, index=False)